# ChromaDB Migration

**Use case:** You have a ChromaDB collection embedded with one model and want to migrate it to a new model's embedding space without re-embedding every document.

**When you'd reach for this:** Your ChromaDB corpus is large (100K+ documents), re-embedding is expensive/slow, and you want to start using a better embedding model today.

**What you need installed:** `isotrieve`, `chromadb`, `numpy`.

**Estimated runtime:** ~2 minutes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krish1925/AECP/blob/main/isotrieve-python/notebooks/02_chromadb_migration.ipynb)

In [ ]:
!pip install -q isotrieve chromadb numpy scikit-learn

In [ ]:
import numpy as np
import chromadb
from isotrieve import RidgeMapping
from isotrieve.adapters.chroma import migrate_collection
from isotrieve.quality.gate import QualityGate

print("imports OK")

## 1. Set up a ChromaDB collection with "old model" vectors

We simulate an existing ChromaDB collection that was embedded with a 384-dim model. In production, this would be your real collection.

In [ ]:
rng = np.random.default_rng(42)

N_DOCS = 500
D_OLD = 384
D_NEW = 768
LATENT = 64

# Generate synthetic corpus with known structure
latent = rng.normal(size=(N_DOCS, LATENT))
W_old = rng.normal(size=(LATENT, D_OLD)) / np.sqrt(LATENT)
W_new = rng.normal(size=(LATENT, D_NEW)) / np.sqrt(LATENT)

doc_vectors_old = latent @ W_old
doc_vectors_old = doc_vectors_old / np.linalg.norm(doc_vectors_old, axis=1, keepdims=True)

doc_vectors_new = latent @ W_new
doc_vectors_new = doc_vectors_new / np.linalg.norm(doc_vectors_new, axis=1, keepdims=True)

# Create fake documents
documents = [f"Document {i} about topic {i % 20}" for i in range(N_DOCS)]
ids = [f"doc_{i}" for i in range(N_DOCS)]

print(f"Corpus: {N_DOCS} documents, {D_OLD}-dim (old model)")

In [ ]:
# Create ephemeral ChromaDB and populate with old-model vectors
client = chromadb.Client()
collection = client.create_collection(
    name="my_docs",
    metadata={"hnsw:space": "cosine"},
)

collection.add(
    ids=ids,
    embeddings=doc_vectors_old.tolist(),
    documents=documents,
)

print(f"ChromaDB collection '{collection.name}': {collection.count()} vectors")

## 2. Fit the mapping (old model -> new model)

We need a calibration set: texts embedded with both old and new models. In practice, you'd pick ~1000-2000 representative texts from your domain.

In [ ]:
# Calibration set (separate from the corpus)
N_CAL = 1500
latent_cal = rng.normal(size=(N_CAL, LATENT))
X_cal = (latent_cal @ W_old)
X_cal = X_cal / np.linalg.norm(X_cal, axis=1, keepdims=True)
Y_cal = (latent_cal @ W_new)
Y_cal = Y_cal / np.linalg.norm(Y_cal, axis=1, keepdims=True)

mapping = RidgeMapping(alpha="auto", seed=0)
mapping.fit(X_cal, Y_cal)

vreport = mapping.validation_report()
print(f"Mapping fitted: {mapping.d_src} -> {mapping.d_tgt}")
print(f"Holdout cosine mean: {vreport.holdout_cosine_mean:.4f}")
print(f"Top-1 retention: {vreport.top1_retention:.4f}")

## 3. Migrate the ChromaDB collection

This transforms every vector in the collection from old-space to new-space and writes to a new collection. The original collection is never modified.

In [ ]:
report = migrate_collection(
    client,
    collection_name="my_docs",
    mapping=mapping,
    new_collection="my_docs_new_model",
)

print(f"Rows processed: {report.rows_processed}")
print(f"Source: {report.source_collection}")
print(f"Target: {report.target_collection}")
print(f"Errors: {len(report.errors)}")

## 4. Compare retrieval quality

We measure top-k overlap: how many of the top results from the migrated collection match the top results from a full re-embed baseline.

In [ ]:
# Generate queries using the NEW model
N_QUERIES = 20
latent_q = rng.normal(size=(N_QUERIES, LATENT))
Q_new = (latent_q @ W_new)
Q_new = Q_new / np.linalg.norm(Q_new, axis=1, keepdims=True)

# Ground truth: search with new-model vectors directly
ground_truth_collection = client.create_collection(
    name="ground_truth",
    metadata={"hnsw:space": "cosine"},
)
ground_truth_collection.add(
    ids=ids,
    embeddings=doc_vectors_new.tolist(),
)

# Search ground truth
gt_results = ground_truth_collection.query(
    query_embeddings=Q_new.tolist(),
    n_results=10,
)

# Search migrated collection
# After migration, collection has d_new vectors, so queries must also be d_new
migrated_collection = client.get_collection("my_docs_new_model")
migrated_results = migrated_collection.query(
    query_embeddings=Q_new.tolist(),
    n_results=10,
)

# Compute top-k overlap
overlaps = []
for gt_ids, mig_ids in zip(gt_results["ids"], migrated_results["ids"]):
    overlap = len(set(gt_ids) & set(mig_ids))
    overlaps.append(overlap)

mean_overlap = np.mean(overlaps)
print(f"Top-10 overlap (migrated vs full re-embed): {mean_overlap:.1f}/10")
print(f"Retention: {mean_overlap/10*100:.0f}%")

## 5. Query-time transformation (no migration needed)

Instead of migrating the whole collection, you can transform queries at search time. Embed with the new model, inverse-map to old space, and search the original collection.

In [ ]:
# A new query comes in from the new model
new_query = rng.normal(size=D_NEW)
new_query = new_query / np.linalg.norm(new_query)

# Inverse-map to old space
old_space_query = mapping.inverse_transform(new_query.reshape(1, -1)).ravel()

# Search the ORIGINAL (un-migrated) collection with the old-space query
qt_results = collection.query(
    query_embeddings=[old_space_query.tolist()],
    n_results=3,
)

print(f"Query-time transform: new-space ({D_NEW}d) -> old-space ({D_OLD}d)")
print(f"Top-3 results from original collection: {qt_results['ids'][0]}")
print(f"Scores: {[f'{s:.4f}' for s in qt_results['distances'][0]]}")
print(f"\nNo migration was needed — just a matrix multiply per query.")

## 6. Run the quality gate on the mapping

Before deploying to production, run the gate on held-out data.

In [ ]:
# Evaluation set (disjoint from calibration)
X_eval = (latent[:200] @ W_old)
X_eval = X_eval / np.linalg.norm(X_eval, axis=1, keepdims=True)
Y_eval = (latent[:200] @ W_new)
Y_eval = Y_eval / np.linalg.norm(Y_eval, axis=1, keepdims=True)

gate = QualityGate()
gate_report = gate.evaluate(mapping, X_eval, Y_eval)

print(f"Gate verdict: {gate_report.verdict.value}")
print(f"Predicted retention: {gate_report.predicted_retention:.3f}")
print(f"Margin compression: {gate_report.margin_compression:.3f}" if gate_report.margin_compression else "")

## Summary

| Approach | Cost | Quality |
|----------|------|---------|
| Re-embed everything | O(N) embedding calls | 100% (baseline) |
| isotrieve migration | O(K) calibration + O(N) matrix multiply | ~90% retention |
| Do nothing | 0 | 0% (old model, can't use new model at all) |

The migrated collection works with the new model's query space. No re-embedding of the corpus was needed.

## Try it yourself

Change `N_CAL` (calibration set size) from 1500 to 200. Does the gate verdict change? What's the minimum calibration size that still gives PASS?